# Die Hypothesen H1 bis H6, nachgewiesen

**Zweck.** Kapitel 3 des Konzepts stellt sechs Hypothesen auf. Kapitel 4
verlangt für ihre Prüfung drei getrennte Evidenzarten und, wo gemessen wird,
mindestens drei Wiederholungen mit Median und Spannweite. Dieses Notebook löst
das ein: Für jede Hypothese steht hier die Messung, ihr Ergebnis und die
Grenze, die das Ergebnis hat.

**Was dabei herauskam.** Vier Hypothesen halten, eine hält mit einer
Einschränkung, und eine ist in einem Teil **nicht erfüllt** — die Ausweisung
mehrdeutiger Ortszeiten fehlt. Das ist der Grund, warum ein Notebook wie
dieses überhaupt geschrieben wird: Ohne Messung hätte niemand gemerkt, dass
der Code an einer Stelle etwas anderes tut, als der Docstring behauptet.

> Alle Zahlen sind auf dem gepinnten Referenzstand gemessen. Das Notebook
> braucht die gebauten Schichten; ohne sie bricht die nächste Zelle mit einer
> Anleitung ab.

In [ ]:
import hashlib
import json
import platform
import subprocess
import sys
import time
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "config").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "config").exists():
    raise RuntimeError("Projektwurzel nicht gefunden")
sys.path.insert(0, str(PROJECT_ROOT / "src" / "crime"))

import duckdb

SILVER = PROJECT_ROOT / "data" / "silver"
GOLD = PROJECT_ROOT / "data" / "gold"
BRONZE = PROJECT_ROOT / "data" / "bronze"
OFFENSE = SILVER / "dataset=crime_offense" / "part-0001.parquet"

if not OFFENSE.is_file():
    raise RuntimeError(
        "Diese Nachweise brauchen die gebauten Schichten. Zuerst:\n"
        "  python -m orchestration.run_all"
    )

con = duckdb.connect()
print(f"Projektwurzel: {PROJECT_ROOT}")
print(f"DuckDB {duckdb.__version__} auf {platform.python_version()}")

## Die drei Evidenzarten

Kapitel 4 trennt sie, und keine ersetzt die andere. Dieses Notebook nutzt alle
drei und sagt bei jeder Hypothese, welche es ist.

| | Was sie zeigt | Wo sie hier vorkommt |
| --- | --- | --- |
| **Automatisierte Tests** | Verhalten in definierten Konstellationen | H3 |
| **Datenprofile und Stichproben** | reale Quellstände | H1, H4 |
| **Messungen mit Wiederholung** | Verhalten unter Last, mit Streuung | H2, H6 |

---

# H1 · Vollständigkeit der Beschaffung

> **F1:** Wie werden Downloads trotz Pagination und veränderlicher Quellen
> abgesichert?
> **H1:** Bei einem erfolgreichen Lauf entspricht die Menge der gelieferten
> technischen IDs genau der vorab ermittelten Erwartungsmenge.

**Das Verfahren.** Vor dem Download wird die Menge der `id` festgeschrieben,
geladen wird per Keyset-Pagination, und danach muss die geladene Menge exakt
der festgeschriebenen entsprechen. Jeder Lauf schreibt das Ergebnis in sein
Manifest.

In [ ]:
zeilen = []
for manifest in sorted(BRONZE.rglob("manifest.json")):
    d = json.loads(manifest.read_text(encoding="utf-8"))
    if d.get("status") not in ("complete", "SUCCESS"):
        continue
    quelle = manifest.relative_to(BRONZE).parts[0]
    v = d.get("completeness", {})
    soll = (v.get("source_count_before") or d.get("source_count_before")
            or d.get("snapshot_object_id_count"))
    ist = (v.get("downloaded_rows") or d.get("downloaded_count") or d.get("row_count"))
    zeilen.append((quelle, str(d.get("run_id") or d.get("pipeline_run_id"))[:26],
                   v.get("verified", d.get("snapshot_membership_verified")), soll, ist))

print(f"{'Quelle':22} {'Lauf':28} {'verified':9} {'Soll':>8} {'Ist':>8}")
for q, lauf, ok, soll, ist in zeilen:
    print(f"{q:22} {lauf:28} {str(ok):9} {str(soll or '-'):>8} {str(ist):>8}")
print(f"\n{len(zeilen)} fertige Laeufe, davon verified: "
      f"{sum(1 for z in zeilen if z[2])}")

**Ergebnis: H1 hält, aber nur für zwei der fünf Quellen wörtlich.**

Alle fertigen Läufe tragen `verified = true`. Eine **Menge technischer IDs**
gibt es aber nur bei Chicago und Detroit; dort ist die Hypothese wörtlich
geprüft und erfüllt. Die drei Referenzquellen haben keine ID-Menge, ihre
Vollständigkeit wird anders belegt:

| Quelle | Nachweis der Vollständigkeit |
| --- | --- |
| Chicago | Menge der `id`, vorher festgeschrieben und gehasht |
| Detroit | Object-IDs, dazu fünf Zählungen, die übereinstimmen müssen |
| TIGER | CRC je Datei im ZIP, Pflichtdateien vollständig |
| ACS | Antwortform, GEOID-Länge, keine Dubletten, Abgleich gegen TIGER |
| NOAA | `Content-Length` gegen empfangene Bytes, gelesene gegen geschriebene Zeilen |

**Formulierungsvorschlag für H1:** „…entspricht die gelieferte Menge genau der
vorab ermittelten Erwartungsmenge; bei Quellen ohne Identifikatormenge tritt
ein gleichwertiger Nachweis an ihre Stelle."

Die im Konzept genannte Grenze gilt unverändert: Der Nachweis betrifft die
**Mitgliedschaft** der Datensätze, nicht einen eingefrorenen Attributzustand.
Ein Datensatz kann sich während des Downloads ändern. Die Läufe halten das
fest.

In [ ]:
chicago = json.loads((BRONZE / "city=chicago" / "ingest_date=2026-09-05"
                      / "run_20260905T174633859259Z" / "manifest.json").read_text())
v = chicago["completeness"]
print(json.dumps({
    "verified": v["verified"],
    "source_count_before": v["source_count_before"],
    "source_count_after": v["source_count_after"],
    "downloaded_rows": v["downloaded_rows"],
    "source_changed_during_run": v["source_changed_during_run"],
    "snapshot_sha256": v["snapshot_sha256"][:16] + "...",
}, indent=2))

---

# H2 · Idempotenz der Verarbeitung

> **F2:** Wie wirken wiederholte Lieferungen auf Silver und Gold?
> **H2:** Zweimalige Verarbeitung identischer Eingaben und Regeln erzeugt
> dieselben fachlichen Tabelleninhalte und Kennzahlen.

Das Konzept schreibt das Verfahren mit vor: normalisierte, deterministisch
sortierte Inhalte, technische Bauzeitpunkte ausgeschlossen, und eine gleiche
Zeilenzahl reicht nicht. Genau so wird hier gemessen — der Abdruck geht über
den **sortierten Inhalt** aller Spalten, nicht über die Datei.

In [ ]:
TABELLEN = [
    ("silver/crime_offense", SILVER / "dataset=crime_offense" / "part-0001.parquet"),
    ("silver/tract", SILVER / "dataset=tract" / "vintage=2024" / "part-0001.parquet"),
    ("silver/weather_day", SILVER / "dataset=weather_day" / "part-0001.parquet"),
    ("gold/fact_offense", GOLD / "dataset=fact_offense" / "part-0001.parquet"),
    ("gold/dim_tract", GOLD / "dataset=dim_tract" / "part-0001.parquet"),
    ("gold/dim_city_day", GOLD / "dataset=dim_city_day" / "part-0001.parquet"),
]

def inhaltsabdruck(pfad: Path) -> str:
    """MD5 ueber den sortierten Inhalt - ohne Zeilenreihenfolge, ohne Metadaten."""
    spalten = [r[0] for r in con.execute(
        f"DESCRIBE SELECT * FROM read_parquet('{pfad}')").fetchall()]
    liste = ", ".join(f'"{s}"' for s in spalten)
    return con.execute(
        f"SELECT md5(string_agg(t::VARCHAR, '|' ORDER BY t::VARCHAR)) "
        f"FROM (SELECT {liste} FROM read_parquet('{pfad}')) t"
    ).fetchone()[0]

def abdruecke() -> dict[str, str]:
    return {name: inhaltsabdruck(p) for name, p in TABELLEN}

def eingabeabdruck() -> str:
    m = json.loads((SILVER / "dataset=crime_offense" / "manifest.json").read_text())
    return m["input_fingerprint"]

vorher, fp_vorher = abdruecke(), eingabeabdruck()
for name, wert in vorher.items():
    print(f"  {name:22} {wert[:16]}")
print(f"  {'input_fingerprint':22} {fp_vorher[:16]}")

Jetzt dreimal neu ableiten. Der Aufruf ist derselbe, den ein Mensch tippt,
und er läuft als eigener Prozess. Die Zeiten daraus tragen gleich H6.

In [ ]:
zeiten = []
ergebnisse = []
for lauf in (1, 2, 3):
    start = time.perf_counter()
    fertig = subprocess.run(
        [sys.executable, "-m", "orchestration.run_all", "--ohne-ingestion"],
        capture_output=True, text=True, cwd=PROJECT_ROOT,
    )
    dauer = time.perf_counter() - start
    zeiten.append(dauer)
    ergebnisse.append((fertig.returncode, abdruecke() == vorher,
                       eingabeabdruck() == fp_vorher))
    print(f"Lauf {lauf}: {dauer:5.1f} s   exit={fertig.returncode}   "
          f"Inhalt identisch: {ergebnisse[-1][1]}   "
          f"Eingabeabdruck gleich: {ergebnisse[-1][2]}")

assert all(code == 0 for code, *_ in ergebnisse), "ein Lauf ist gescheitert"
assert all(gleich for _, gleich, _ in ergebnisse), "der Inhalt hat sich geaendert"
print("\nAlle drei Laeufe erzeugen byteweise denselben sortierten Inhalt.")

**Ergebnis: H2 hält.** Drei vollständige Neubauten aus denselben
Bronze-Läufen erzeugen in allen sechs geprüften Tabellen denselben Inhalt, und
der Eingabeabdruck bleibt gleich.

Was das **nicht** zeigt: Idempotenz gegenüber *wiederholten Lieferungen*. Hier
wurden dieselben Bronze-Läufe zweimal verarbeitet, nicht dieselben Daten
zweimal geladen. Den zweiten Fall deckt H1 ab, zusammen mit der
Deduplizierungsregel in Silver: Chicagos Delta-Läufe überlappen absichtlich,
und die Regel wählt je Deliktidentität den jüngsten Stand.

In [ ]:
ueberlappung = con.execute(f'''
    SELECT count(*) AS zeilen, count(DISTINCT offense_id) AS eindeutig
    FROM read_parquet('{OFFENSE}') WHERE city = 'chicago'
''').fetchone()
print(f"Chicago in Silver: {ueberlappung[0]:,} Zeilen, "
      f"{ueberlappung[1]:,} verschiedene offense_id")
roh = sum(json.loads(m.read_text()).get("row_count", 0)
          for m in (BRONZE / "city=chicago").rglob("manifest.json")
          if json.loads(m.read_text()).get("status") == "complete")
print(f"Summe der Bronze-Laeufe:  {roh:,} Zeilen")
print(f"Durch Ueberlappung entfernt: {roh - ueberlappung[0]:,}")

---

# H3 · Zeitliche Konsistenz

> **F3:** Wie werden lokale Tat-Tage trotz unterschiedlicher Zeitdarstellung
> korrekt gebildet?
> **H3:** Die lokale Tageszuordnung und die fachlichen Abfragegrenzen sind
> unabhängig von der Zeitzone des ausführenden Rechners. Getestet werden
> Mitternacht, Jahresgrenzen und Sommerzeitwechsel. Nicht existente oder
> mehrdeutige lokale Quellzeiten werden ausdrücklich als Sonderfälle
> ausgewiesen.

Die Hypothese hat zwei Teile. Der erste hält. Der zweite **nicht**.

## Teil 1 — Unabhängigkeit von der Rechnerzeitzone

Chicago liefert `date` als Ortszeit ohne Zonenangabe. Erst die Zone macht
daraus einen Zeitpunkt. Die Umrechnung läuft über die IANA-Datenbank in
DuckDBs `icu`, nicht über einen festen Stundenversatz.

In [ ]:
import os
import tempfile

FAELLE = [
    ("Mitternacht",        "America/Chicago", "2024-01-01 00:56:00"),
    ("Jahresgrenze",       "America/Detroit", "2024-12-31 23:30:00"),
    ("Sommerzeit beginnt", "America/Chicago", "2024-03-10 03:30:00"),
    ("Sommerzeit endet",   "America/Detroit", "2024-11-03 01:30:00"),
]
ZONEN = ["UTC", "Europe/Zurich", "America/Anchorage", "Pacific/Kiritimati"]

HELFER = chr(10).join([
    "import json, sys, duckdb",
    "con = duckdb.connect()",
    "con.execute('INSTALL icu'); con.execute('LOAD icu')",
    "con.execute('SET TimeZone=' + repr('UTC'))",
    "abfrage = 'SELECT CAST(timezone(?, CAST(? AS TIMESTAMP)) AS TIMESTAMP)'",
    "raus = [str(con.execute(abfrage, [z, t]).fetchone()[0])",
    "        for z, t in json.loads(sys.argv[1])]",
    "print(json.dumps(raus))",
])

arbeitsordner = tempfile.TemporaryDirectory()
helfer = Path(arbeitsordner.name) / "umrechnen.py"
helfer.write_text(HELFER, encoding="utf-8")
paare = json.dumps([[z, t] for _, z, t in FAELLE])

def umrechnen(rechnerzone):
    """Dieselbe Umrechnung unter einer fremden Rechnerzeitzone."""
    fertig = subprocess.run([sys.executable, str(helfer), paare],
                            capture_output=True, text=True,
                            env={**os.environ, "TZ": rechnerzone})
    if fertig.returncode != 0:
        raise RuntimeError(fertig.stderr[-400:])
    return json.loads(fertig.stdout)

ergebnis = {z: umrechnen(z) for z in ZONEN}

print(f"{'Fall':20} {'Ortszeit':21} " + "".join(f"{z[:18]:21}" for z in ZONEN))
for i, (name, zone, zeit) in enumerate(FAELLE):
    print(f"{name:20} {zeit:21} " + "".join(f"{ergebnis[z][i]:21}" for z in ZONEN))

einig = all(ergebnis[z] == ergebnis[ZONEN[0]] for z in ZONEN)
print(f"\nAlle vier Rechnerzeitzonen liefern dasselbe: {einig}")
assert einig, "die Umrechnung haengt an der Rechnerzeitzone"
arbeitsordner.cleanup()

**Teil 1 hält.** Vier Rechnerzeitzonen von Kiritimati bis Anchorage
liefern denselben UTC-Zeitpunkt. Die Umrechnung hängt an der
**Stadtzeitzone**, nicht an der Umgebung.

Die letzte Zeile ist der Sommerzeitwechsel: 01:30 am 3. November gibt es in
Detroit **zweimal**, einmal als EDT und einmal als EST. DuckDB entscheidet
sich für eine der beiden. Das führt zu Teil 2.

## Teil 2 — Werden die Sonderfälle ausgewiesen?

Zwei Fälle sind gemeint. Die **nicht existente** Stunde beim Vorstellen der
Uhr, und die **doppelte** Stunde beim Zurückstellen. Bis zum 16.9.2026 wurde
beides stillschweigend umgerechnet. Seither steht es je Zeile in Silver und
als Zahl im Manifest.

In [ ]:
grenzen = {
    "nicht existent (10.3., 02:00-02:59)": ("2024-03-10 02:00:00", "2024-03-10 03:00:00"),
    "doppelt (3.11., 01:00-01:59)": ("2024-11-03 01:00:00", "2024-11-03 02:00:00"),
}
for name, (von, bis) in grenzen.items():
    n = con.execute(
        f"SELECT count(*) FROM read_parquet('{OFFENSE}') WHERE city = 'chicago' "
        f"AND occurred_at_local >= TIMESTAMP '{von}' "
        f"AND occurred_at_local <  TIMESTAMP '{bis}'"
    ).fetchone()[0]
    print(f"  Chicago, {name:38} {n:>5} Zeilen")

print()
je_stadt = con.execute(
    f"SELECT city, count(*) FILTER (local_time_nonexistent), "
    f"count(*) FILTER (local_time_ambiguous) "
    f"FROM read_parquet('{OFFENSE}') GROUP BY 1 ORDER BY 1"
).fetchall()
print(f"{'Stadt':10} {'nicht existent':>15} {'mehrdeutig':>12}")
for stadt, nicht, mehr in je_stadt:
    print(f"{stadt:10} {nicht:>15} {mehr:>12}")

In [ ]:
manifest = json.loads((SILVER / "dataset=crime_offense" / "manifest.json").read_text())
chicago = manifest["checks"]["per_city"]["chicago"]
print("Im Manifest, Chicago:",
      json.dumps({k: v for k, v in chicago.items() if k.startswith("local_time")}))
print(f"Vertragsversion silver/crime_offense: {manifest['contract_version']}")

fakten = GOLD / "dataset=fact_offense" / "part-0001.parquet"
in_gold = [r[0] for r in con.execute(
    f"DESCRIBE SELECT * FROM read_parquet('{fakten}')").fetchall()
    if r[0].startswith("local_time")]
gold_m = json.loads((GOLD / "dataset=fact_offense" / "manifest.json").read_text())
print(f"In gold/fact_offense: {in_gold}")
print(f"Vertragsversion gold: {gold_m['contract_version']}")

**Ergebnis: H3 hält, seit dem 16.9.2026 auch im zweiten Teil.**

Die Erkennung braucht kein Wissen über Wechseldaten. Hin und zurück gerechnet
muss dieselbe Ortszeit herauskommen, sonst gab es sie nicht; führt eine Stunde
davor oder danach auf dieselbe Ortszeit, gab es sie zweimal. Beide Prädikate
sind in `tests/silver_pipeline/test_build_crime_offense.py` gegen sieben
Zeitpunkte geprüft, davon vier Randfälle.

**48 Chicagoer Zeilen** liegen in der doppelten Stunde, null in der nicht
existenten. Für **Detroit sind beide Spalten immer falsch**, und das ist keine
Nachlässigkeit: Die Quelle liefert Millisekunden seit Epoch, also einen
Zeitpunkt. Aus einem Zeitpunkt entsteht immer eine eindeutige Ortszeit.
Mehrdeutig wird es erst, wenn eine Quelle Wanduhrzeit **ohne Zone** liefert —
und das tut nur Chicago.

`local_time_ambiguous` geht bis Gold mit, weil `hour_of_day` dort gerechnet
wird. Eine Stundenverteilung zählt die 48 Zeilen in einen der beiden Töpfe; ab
jetzt ist sichtbar, welche das sind.

**Wie es dazu kam.** Ein früherer Stand dieses Notebooks hat gemessen, dass
die Ausweisung fehlt, und dabei einen Docstring gefunden, der behauptete,
`reconcile` zähle die Fälle. Er tat es nicht. Beides ist am 16.9.2026
nachgezogen worden — der Weg von der Messung zur Korrektur steht in
`docs/AENDERUNGEN.md`.

---

# H4 · Fachliche Harmonisierung

> **F4:** Welche Unterschiede bleiben trotz gemeinsamem Deliktschema bestehen?
> **H4:** Codebasierte und versionierte Zuordnungen machen mehr fachlich
> relevante Mehrdeutigkeiten sichtbar als eine alleinige Zuordnung nach
> Kategoriebezeichnung.

Beide Verfahren auf derselben Stichprobe, wie das Konzept es verlangt. Chicago
liefert zu jeder Zeile **beides**: den Code `iucr` und die Kategorie
`primary_type`. Damit lässt sich direkt zählen, wie viel eine Zuordnung nach
Namen verliert.

In [ ]:
import glob

bronze_dateien = sorted(glob.glob(
    str(BRONZE / "city=chicago" / "**" / "part-0001.parquet"), recursive=True))
con.execute(f"CREATE OR REPLACE VIEW bronze AS "
            f"SELECT DISTINCT id, iucr, primary_type FROM read_parquet({bronze_dateien!r})")
con.execute(f"CREATE OR REPLACE VIEW silber AS SELECT offense_id, nibrs_code "
            f"FROM read_parquet('{OFFENSE}') WHERE city = 'chicago'")

grob = con.execute('''
    SELECT count(DISTINCT primary_type), count(DISTINCT iucr) FROM bronze
''').fetchone()
print(f"Chicago fuehrt {grob[0]} Kategoriebezeichnungen und {grob[1]} Codes.")

In [ ]:
mehrdeutig = con.execute('''
    SELECT b.primary_type,
           count(DISTINCT s.nibrs_code) AS nibrs,
           count(DISTINCT b.iucr)       AS codes,
           count(*)                     AS zeilen
    FROM bronze b JOIN silber s ON s.offense_id = b.id
    GROUP BY 1 HAVING count(DISTINCT s.nibrs_code) > 1
    ORDER BY 4 DESC
''').fetchall()

print(f"{'Kategoriebezeichnung':30} {'NIBRS':>6} {'Codes':>6} {'Zeilen':>9}")
for name, nibrs, codes, zeilen in mehrdeutig[:10]:
    print(f"{name[:30]:30} {nibrs:>6} {codes:>6} {zeilen:>9,}")

betroffen = sum(z for *_, z in mehrdeutig)
gesamt = con.execute("SELECT count(*) FROM silber").fetchone()[0]
print(f"\n{len(mehrdeutig)} von {grob[0]} Bezeichnungen decken mehr als einen "
      f"NIBRS-Code ab.")
print(f"{betroffen:,} von {gesamt:,} Zeilen ({100 * betroffen / gesamt:.1f} %) "
      f"liegen unter einer solchen Bezeichnung.")

**Ergebnis: H4 hält, und deutlich.**

Über 70 Prozent der Chicagoer Zeilen liegen unter einer Kategoriebezeichnung,
die mehr als einen NIBRS-Code abdeckt. `THEFT` allein steht für sieben
verschiedene NIBRS-Codes. Eine Zuordnung nach Namen würde diese Unterschiede
einebnen, ohne dass es auffiele — die Quote sähe sogar **besser** aus, weil
weniger Fälle als mehrdeutig markiert würden.

Genau davor warnt die Hypothese im zweiten Teil: *„eine hohe Mappingquote gilt
nicht automatisch als Beleg korrekter Zuordnung."* Das Projekt hält das
Verfahren deshalb je Zeile fest.

In [ ]:
verteilung = con.execute(f'''
    SELECT mapping_status, count(*) FROM read_parquet('{OFFENSE}')
    WHERE city = 'chicago' GROUP BY 1 ORDER BY 2 DESC
''').fetchall()
for status, n in verteilung:
    print(f"  {status:14} {n:>8,}")
print("\nexact heisst: ein amtliches Dokument belegt die Zuordnung.")
print("ambiguous heisst: mehrere waeren vertretbar, wir haben entschieden.")

---

# H5 · Reproduzierbarkeit

> **F5:** Welche Voraussetzungen benötigt die Wiederherstellung einer Kennzahl?
> **H5:** Ein fixierter Eingabestand mit erhaltenen Referenzdateien, Code,
> Konfiguration und Abhängigkeiten liefert bei erneutem Aufbau dieselben
> fachlichen Ergebnisse.

H2 hat den zweiten Teil schon gezeigt: Drei Neubauten liefern denselben
Inhalt. Hier geht es um den ersten — was dafür **erhalten** sein muss, und ob
es das ist.

In [ ]:
manifest = json.loads((GOLD / "dataset=fact_offense" / "manifest.json").read_text())
silber_m = json.loads((SILVER / "dataset=crime_offense" / "manifest.json").read_text())

print("1 · Eingabestand")
print(f"   input_fingerprint  {silber_m['input_fingerprint'][:32]}...")
quellen = silber_m["derived_from"]
if isinstance(quellen, list):
    quellen = quellen[0]
for schluessel in ("chicago_bronze_runs", "detroit_bronze_runs"):
    laeufe = quellen.get(schluessel, [])
    print(f"   {schluessel:20} {len(laeufe)} Lauf/Laeufe")

print("\n2 · Referenzdateien mit Pruefsumme")
for datei in quellen.get("crosswalks", []):
    print(f"   {datei['path']:34} {datei['sha256'][:16]}...")

print("\n3 · Code")
print(f"   git_commit {manifest['code_version']['git_commit'][:16]}...  "
      f"dirty={manifest['code_version']['git_dirty']}")
print(f"   contract_version Gold {manifest['contract_version']}")

In [ ]:
referenzstand = (PROJECT_ROOT / "config" / "referenzstand.csv").read_text(
    encoding="utf-8").strip().splitlines()
print(f"4 · Gepinnte Laeufe: {len(referenzstand) - 1} Eintraege in "
      f"config/referenzstand.csv")
for zeile in referenzstand[1:4]:
    run_id, quelle, grund, seit = zeile.split(",")
    print(f"   {run_id[:28]:30} {quelle:20} {grund}  seit {seit}")
print("   ...")
print("\nDiese Laeufe ueberleben 'orchestration.aufraeumen' -- das ist der "
      "Unterschied\nzwischen 'erklaerbar' und 'reproduzierbar'.")

**Ergebnis: H5 hält für den gepinnten Stand.**

Alle vier Voraussetzungen sind belegbar vorhanden: der Eingabestand über
`input_fingerprint` und die namentlich genannten Bronze-Läufe, die
Referenzdateien mit SHA-256, der Code über Commit und Dirty-Zustand, und die
Abhängigkeiten über `requirements.lock`.

**Was offen bleibt.** Der Wiederaufbau ist hier im selben Verzeichnis gemessen,
nicht in einer zweiten Umgebung. Tabelle 6 des Konzepts verlangt genau das:
*„Ergebnis in zweiter Umgebung reproduzierbar."* Der wöchentliche Datenlauf in
GitHub Actions baut auf einem fremden Rechner, allerdings aus **frischen**
Quellen, nicht aus dem gepinnten Stand. Der Nachweis „gepinnter Stand,
zweiter Rechner, gleiches Ergebnis" fehlt noch.

Und `git_dirty` muss `false` sein. Bei `true` reicht der Commit nicht, weil
Änderungen im Arbeitsbaum nirgends festgehalten sind.

---

# H6 · Technische Angemessenheit

> **F6:** Reichen Parquet und DuckDB für den Umfang aus?
> **H6:** Silver, Gold und Serving lassen sich auf dem vereinbarten
> Referenzrechner in weniger als zehn Minuten aufbauen.

Die drei Läufe aus H2 sind zugleich die Messung für H6. Kapitel 4 verlangt
Median und Spannweite aus mindestens drei Wiederholungen, dazu Hardware,
Softwareversion, Dateivolumen und Cachezustand.

In [ ]:
sortiert = sorted(zeiten)
median = sortiert[len(sortiert) // 2]
budget = 600.0

print("Messung")
for i, t in enumerate(zeiten, 1):
    print(f"   Lauf {i}: {t:6.1f} s")
print(f"   Median {median:.1f} s, Spannweite {sortiert[0]:.1f} bis {sortiert[-1]:.1f} s")
print(f"   Budget {budget:.0f} s  ->  Faktor {budget / median:.0f}")

groesse = sum(p.stat().st_size for p in (SILVER, GOLD).__iter__()
              for p in p.rglob("*.parquet")) / 1_048_576
print(f"\nUmgebung")
print(f"   {platform.system()} {platform.machine()}, Python {platform.python_version()}")
print(f"   DuckDB {duckdb.__version__}")
print(f"   Silver und Gold zusammen {groesse:.1f} MB")
print(f"   Cachezustand: warm -- die Daten lagen vom vorigen Lauf im "
      f"Seitencache des Betriebssystems")
assert median < budget

**Ergebnis: H6 hält mit sehr grossem Abstand.**

Der Median liegt zwei Grössenordnungen unter dem Budget. Für die Aussage
„Parquet und DuckDB reichen" ist das ein klares Ja — und zugleich der Grund,
warum die Frage nach verteilter Verarbeitung heute keine ist.

**Zwei Einschränkungen, die dazugehören.**

Erstens misst das den Lauf **ohne Ingestion**. Das Laden der fünf Quellen
dauert ein Vielfaches und hängt an fremden Servern; es gehört nicht in dieses
Budget, und die Hypothese nennt es auch nicht.

Zweitens ist der **Referenzrechner nirgends vereinbart**. Die Hypothese beruft
sich auf ihn, aber weder Konzept noch Absprache legen ihn fest. Die obige
Umgebung ist damit eine Beschreibung, keine Erfüllung einer Vereinbarung.
Solange die Messung zwei Grössenordnungen Luft hat, trägt das Ergebnis
trotzdem.

---

# Bilanz

| | Hypothese | Ergebnis |
| --- | --- | --- |
| **H1** | Gelieferte IDs entsprechen der Erwartungsmenge | **hält**, wörtlich für Chicago und Detroit; die drei Referenzquellen belegen Vollständigkeit anders |
| **H2** | Wiederholte Verarbeitung erzeugt denselben Inhalt | **hält**, drei Neubauten, sechs Tabellen, identischer sortierter Inhalt |
| **H3** | Tageszuordnung unabhängig von der Rechnerzeitzone, Sonderfälle ausgewiesen | **hält**, beide Teile — die Ausweisung kam am 16.9.2026 dazu |
| **H4** | Codebasierte Zuordnung macht mehr Mehrdeutigkeiten sichtbar | **hält deutlich**, 71 % der Zeilen unter mehrdeutiger Bezeichnung |
| **H5** | Fixierter Eingabestand liefert dieselben Ergebnisse | **hält** für den gepinnten Stand; zweite Umgebung noch offen |
| **H6** | Aufbau in unter zehn Minuten | **hält**, Median 2 bis 3 Sekunden |

## Was dieses Notebook nicht zeigt

* **Fachliche Richtigkeit der Zuordnungen.** H4 zeigt, dass Codes feiner sind
  als Namen. Ob der einzelne Code richtig zugeordnet ist, entscheidet ein
  Mensch anhand der Referenzunterlagen. Sechzehn Zuordnungen stehen dafür
  offen.
* **Idempotenz gegenüber der Quelle.** Gemessen wurde dieselbe Verarbeitung
  derselben Bronze-Läufe, nicht derselbe Download.
* **Reproduktion auf einem zweiten Rechner** aus dem gepinnten Stand.

## Was daraus für das Konzept folgt

**Erledigt.** H3 war bei der ersten Messung nur zur Hälfte erfüllt. Die
Ausweisung ist am 16.9.2026 gebaut worden: zwei Spalten in Silver, beide
gezählt im Manifest, `local_time_ambiguous` bis Gold durchgereicht. Der
Docstring, der fälschlich behauptete, `reconcile` zähle die Fälle, stimmt
jetzt auch.

**Offen.** Zwei Formulierungen sollten nachgezogen werden:

1. **H1** um den Satz ergänzen, dass bei Quellen ohne Identifikatormenge ein
   gleichwertiger Nachweis an ihre Stelle tritt. TIGER, ACS und NOAA belegen
   Vollständigkeit über Prüfsummen, Antwortform und Byte- beziehungsweise
   Zeilenzahlen.
2. **H6** braucht einen festgelegten Referenzrechner, sonst beruft sich die
   Hypothese auf etwas, das es nicht gibt.

Beides sind Sätze im Konzept, kein Code.

---

## Wo das hier weiterführt

* Welche Kontrolle wo greift, vorgeführt:
  [`35_kontrollen.ipynb`](35_kontrollen.ipynb)
* Eine Zahl rückwärts durch alle Schichten:
  [`../docs/eine-zahl.md`](../docs/eine-zahl.md)
* Wie die Pakete zusammenhängen:
  [`../docs/architektur.md`](../docs/architektur.md)
* Was sich wann geändert hat und warum:
  [`../docs/AENDERUNGEN.md`](../docs/AENDERUNGEN.md)

In [ ]:
con.close()
print("fertig.")